# 04 — Por que a taxa de atraso é tão alta?

O notebook anterior (`02_kpis`) mostrou **onde** os atrasos acontecem.  
Este notebook investiga o **porquê** — análise de causa raiz em 4 dimensões:

1. **Prazo prometido vs. realidade** — os prazos acordados são factíveis?
2. **Sazonalidade** — há padrões temporais que amplificam os atrasos?
3. **Combinação modal × região** — onde o problema é estrutural?
4. **Volume × capacidade** — picos de demanda travam a operação?
5. **Ranking de fatores** — o que mais explica o atraso (Random Forest)?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='latin-1')
df_validos = df[df['Delivery Status'] != 'Shipping canceled'].copy()

df_validos['Desvio Lead Time'] = df_validos['Days for shipping (real)'] - df_validos['Days for shipment (scheduled)']
df_validos['Atrasado'] = (df_validos['Delivery Status'] == 'Late delivery').astype(int)

df_validos['order date (DateOrders)'] = pd.to_datetime(df_validos['order date (DateOrders)'], errors='coerce')
df_validos['Ano'] = df_validos['order date (DateOrders)'].dt.year
df_validos['Mes'] = df_validos['order date (DateOrders)'].dt.month
df_validos['DiaSemana'] = df_validos['order date (DateOrders)'].dt.dayofweek

print(f'Dataset: {len(df_validos):,} pedidos válidos')
print(f'Taxa de atraso geral: {df_validos["Atrasado"].mean()*100:.1f}%')

## 1. Prazo prometido é factível?
Hipótese: a empresa promete prazos que operacionalmente não consegue cumprir de forma consistente.

In [ ]:
desvio = df_validos['Desvio Lead Time']

fig = px.histogram(
    df_validos, x='Desvio Lead Time', nbins=30,
    color='Delivery Status',
    color_discrete_map={'Late delivery': '#e74c3c', 'Shipping on time': '#2ecc71', 'Advance shipping': '#3498db'},
    title='Distribuição do Desvio de Lead Time (dias reais − dias prometidos)',
    labels={'Desvio Lead Time': 'Desvio (dias)', 'count': 'Pedidos'}
)
fig.add_vline(x=0, line_dash='dash', line_color='black', annotation_text='Prazo exato')
fig.add_vline(x=desvio.mean(), line_dash='dot', line_color='orange', annotation_text=f'Média: {desvio.mean():.1f}d')
fig.update_layout(height=420, barmode='overlay')
fig.update_traces(opacity=0.75)
fig.show()

print(f'Desvio médio geral: {desvio.mean():.2f} dias')
print(f'Pedidos mais tarde que prometido: {(desvio > 0).mean()*100:.1f}%')
print(f'Pedidos no prazo ou antes: {(desvio <= 0).mean()*100:.1f}%')

In [ ]:
atraso_por_prazo = df_validos.groupby('Days for shipment (scheduled)').agg(
    Total=('Atrasado', 'count'),
    Taxa_Atraso=('Atrasado', 'mean')
).reset_index()
atraso_por_prazo['Taxa_Atraso_Pct'] = (atraso_por_prazo['Taxa_Atraso'] * 100).round(1)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=atraso_por_prazo['Days for shipment (scheduled)'], y=atraso_por_prazo['Total'],
    name='Volume de Pedidos', marker_color='#bdc3c7'
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=atraso_por_prazo['Days for shipment (scheduled)'], y=atraso_por_prazo['Taxa_Atraso_Pct'],
    name='Taxa de Atraso (%)', mode='lines+markers',
    line=dict(color='#e74c3c', width=3), marker=dict(size=8)
), secondary_y=True)
fig.update_layout(title='Prazo Prometido (dias) × Taxa de Atraso', xaxis_title='Dias Prometidos', height=420)
fig.update_yaxes(title_text='Nº de Pedidos', secondary_y=False)
fig.update_yaxes(title_text='Taxa de Atraso (%)', secondary_y=True, range=[0, 100])
fig.show()

## 2. Sazonalidade — há meses ou períodos críticos?

In [ ]:
sazonal = df_validos.groupby(['Ano', 'Mes']).agg(
    Taxa_Atraso=('Atrasado', 'mean'), Volume=('Atrasado', 'count')
).reset_index()
sazonal['Taxa_Atraso_Pct'] = (sazonal['Taxa_Atraso'] * 100).round(1)
sazonal['Periodo'] = sazonal['Ano'].astype(str) + '-' + sazonal['Mes'].astype(str).str.zfill(2)
sazonal = sazonal.sort_values('Periodo')

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=sazonal['Periodo'], y=sazonal['Volume'],
    name='Volume de Pedidos', marker_color='#d5e8ff', opacity=0.8
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=sazonal['Periodo'], y=sazonal['Taxa_Atraso_Pct'],
    name='Taxa de Atraso (%)', mode='lines+markers',
    line=dict(color='#e74c3c', width=2), marker=dict(size=5)
), secondary_y=True)
fig.update_layout(
    title='Evolução Temporal: Volume de Pedidos vs. Taxa de Atraso (2015–2018)',
    xaxis_tickangle=-45, height=460
)
fig.update_yaxes(title_text='Nº de Pedidos', secondary_y=False)
fig.update_yaxes(title_text='Taxa de Atraso (%)', secondary_y=True, range=[0, 100])
fig.show()

In [ ]:
nomes_dia = {0:'Segunda',1:'Terça',2:'Quarta',3:'Quinta',4:'Sexta',5:'Sábado',6:'Domingo'}

atraso_dia = df_validos.groupby('DiaSemana').agg(
    Taxa_Atraso=('Atrasado', 'mean'), Volume=('Atrasado', 'count')
).reset_index()
atraso_dia['Taxa_Atraso_Pct'] = (atraso_dia['Taxa_Atraso'] * 100).round(1)
atraso_dia['Dia'] = atraso_dia['DiaSemana'].map(nomes_dia)

fig = px.bar(
    atraso_dia, x='Dia', y='Taxa_Atraso_Pct', text='Taxa_Atraso_Pct',
    color='Taxa_Atraso_Pct', color_continuous_scale='RdYlGn_r',
    title='Taxa de Atraso por Dia da Semana (%)',
    labels={'Dia': 'Dia da Semana', 'Taxa_Atraso_Pct': 'Taxa de Atraso (%)'}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_range=[0, 80])
fig.show()

## 3. Heatmap: Modal × Região — onde o problema é estrutural?

In [ ]:
pivot = df_validos.pivot_table(
    values='Atrasado', index='Order Region', columns='Shipping Mode', aggfunc='mean'
) * 100
pivot = pivot.round(1)

fig = px.imshow(
    pivot, text_auto=True,
    color_continuous_scale='RdYlGn_r', zmin=0, zmax=100,
    title='Taxa de Atraso (%) por Região × Modal de Envio',
    labels={'color': 'Taxa Atraso (%)', 'x': 'Modal', 'y': 'Região'}
)
fig.update_layout(height=600)
fig.show()

pior = pivot.stack().idxmax()
print(f'Combinação mais crítica: {pior[0]} + {pior[1]} — {pivot.loc[pior]:.1f}% de atraso')

## 4. Volume × Capacidade — picos de demanda travam a operação?

In [ ]:
df_validos['Data'] = df_validos['order date (DateOrders)'].dt.date

diario = df_validos.groupby('Data').agg(
    Volume=('Atrasado', 'count'), Taxa_Atraso=('Atrasado', 'mean')
).reset_index()
diario['Taxa_Atraso_Pct'] = diario['Taxa_Atraso'] * 100

correlacao = diario['Volume'].corr(diario['Taxa_Atraso_Pct'])

fig = px.scatter(
    diario, x='Volume', y='Taxa_Atraso_Pct', trendline='ols', opacity=0.5,
    title=f'Volume Diário × Taxa de Atraso (correlação: {correlacao:.2f})',
    labels={'Volume': 'Pedidos no Dia', 'Taxa_Atraso_Pct': 'Taxa de Atraso (%)'}
)
fig.update_layout(height=420)
fig.show()

print(f'Correlação volume × taxa de atraso: {correlacao:.3f}')
if abs(correlacao) > 0.3:
    print('Correlação relevante — picos de demanda impactam o serviço.')
else:
    print('Correlação fraca — o volume sozinho não explica os atrasos.')

## 5. Ranking de fatores — o que mais explica o atraso?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

features = ['Shipping Mode', 'Order Region', 'Days for shipment (scheduled)',
            'Order Item Quantity', 'Mes', 'DiaSemana']

df_model = df_validos[features + ['Atrasado']].dropna()

le = LabelEncoder()
df_enc = df_model.copy()
for col in ['Shipping Mode', 'Order Region']:
    df_enc[col] = le.fit_transform(df_enc[col])

X = df_enc[features]
y = df_enc['Atrasado']

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X, y)

nomes_pt = {
    'Shipping Mode': 'Modal de Envio',
    'Order Region': 'Região do Pedido',
    'Days for shipment (scheduled)': 'Prazo Prometido (dias)',
    'Order Item Quantity': 'Qtd. Itens no Pedido',
    'Mes': 'Mês do Pedido',
    'DiaSemana': 'Dia da Semana'
}

importancias = pd.DataFrame({
    'Fator': [nomes_pt[f] for f in features],
    'Importância': rf.feature_importances_
}).sort_values('Importância', ascending=False)

fig = px.bar(
    importancias.sort_values('Importância'),
    x='Importância', y='Fator', orientation='h',
    text=importancias.sort_values('Importância')['Importância'].apply(lambda x: f'{x:.1%}'),
    color='Importância', color_continuous_scale='Blues',
    title='Importância dos Fatores para Predição de Atraso (Random Forest)',
    labels={'Importância': 'Importância Relativa', 'Fator': ''}
)
fig.update_traces(textposition='outside')
fig.update_layout(height=420, coloraxis_showscale=False)
fig.show()

print('Ranking dos fatores de risco:')
for _, row in importancias.iterrows():
    print(f'  {row["Fator"]}: {row["Importância"]:.1%}')

## Conclusão — Por que a taxa de atraso é tão alta?

| Dimensão | Pergunta respondida |
|---|---|
| **Prazo prometido** | Os prazos acordados condizem com a capacidade real? |
| **Sazonalidade** | Há meses/períodos onde a taxa dispara? |
| **Modal × Região** | Quais combinações concentram o problema estruturalmente? |
| **Volume × Capacidade** | Picos de demanda degradam o nível de serviço? |
| **Ranking de fatores** | Qual variável mais prediz o atraso? |

> **Próximo passo sugerido:** com os fatores identificados, é possível construir um modelo preditivo de risco de atraso em tempo real — sinalizando pedidos problemáticos antes da expedição.